#### Time Series kaggle dataset 기반 예제

- https://www.kaggle.com/datasets/uciml/electric-power-consumption-data-set

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, Subset, Dataset
from sklearn.model_selection import train_test_split
from copy import deepcopy
import torch
import torch.nn as nn

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(1)
if device == 'mps':
    torch.cuda.manual_seed_all(1)
print(device)

mps


In [9]:
df = pd.read_csv('./dataset/household_power_consumption.txt', sep=";", 
                 parse_dates={'dt' : ['Date', 'Time']}, 
                 infer_datetime_format=True, 
                 na_values=['nan', '?'],
                 index_col='dt')
df
df_resample = df.resample('h').mean()
raw_data = df_resample.dropna().copy()

/var/folders/q1/wy16nfjn4sn4s87yr7501dc40000gn/T/ipykernel_51056/3131936194.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv('./dataset/household_power_consumption.txt', sep=";",
/var/folders/q1/wy16nfjn4sn4s87yr7501dc40000gn/T/ipykernel_51056/3131936194.py:1: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv('./dataset/household_power_consumption.txt', sep=";",
/var/folders/q1/wy16nfjn4sn4s87yr7501dc40000gn/T/ipykernel_51056/3131936194.py:1: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.

In [10]:
raw_data.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,0.527778,16.861111
2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,6.716667,16.866667
2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,1.433333,16.683333
2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.000000,16.783333
2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,0.416667,17.216667


In [11]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 34168 entries, 2006-12-16 17:00:00 to 2010-11-26 21:00:00
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Global_active_power    34168 non-null  float64
 1   Global_reactive_power  34168 non-null  float64
 2   Voltage                34168 non-null  float64
 3   Global_intensity       34168 non-null  float64
 4   Sub_metering_1         34168 non-null  float64
 5   Sub_metering_2         34168 non-null  float64
 6   Sub_metering_3         34168 non-null  float64
dtypes: float64(7)
memory usage: 2.1 MB
